In [5]:
import pandas as pd

url = ("https://github.com/DataTalksClub/nyc-tlc-data/releases/download/"
       "yellow/yellow_tripdata_2021-01.csv.gz")

df = pd.read_csv(url, nrows=100)

print(df.head())
print(df.dtypes)
print(df.shape)

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  payment_type  fare_amount  extra  mta_tax  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1           2.10           1                  N           142            43             2          8.0    3.0      0.5   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1           0.20           1                  N           238           151             2          3.0    0.5      0.5   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1          14.70           1                  N           132           165             1         42.0    0.5      0.5   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0          10.60           1                  N           138           132             1         29.0    0.5      0.5   
4         2  2021-01-01 00:31:49   2021-

In [ ]:
import pandas as pd
url = ("https://github.com/DataTalksClub/nyc-tlc-data/releases/download/"
       "yellow/yellow_tripdata_2021-01.csv.gz")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
n_rows = 0
null_counts = pd.Series(dtype="float64")
min_rows, max_rows = [], []
col_dtypes = None
for chunk in pd.read_csv(url, chunksize=100_000):
    if col_dtypes is None:
        col_dtypes = chunk.dtypes.astype(str)
    n_rows += len(chunk)
    null_counts = null_counts.add(chunk.isna().sum(), fill_value=0)
    min_rows.append(chunk.min())
    max_rows.append(chunk.max())
profile = pd.DataFrame({
    "dtype": col_dtypes,
    "nulls": null_counts.astype(int),
    "nulls_%": (null_counts / n_rows * 100).round(2),
    "min": pd.DataFrame(min_rows).min(),
    "max": pd.DataFrame(max_rows).max(),
})
print("Total rows:", n_rows)
print(profile)

/tmp/ipykernel_3141/2272912762.py:14: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(url, chunksize=100_000):


Total rows: 1369765
                         dtype  nulls  nulls_%                  min                  max
DOLocationID             int64      0     0.00                    1                  265
PULocationID             int64      0     0.00                    1                  265
RatecodeID               int64  98352     7.18                  1.0                 99.0
VendorID                 int64  98352     7.18                  1.0                  2.0
congestion_surcharge   float64      0     0.00                 -2.5                  3.0
extra                  float64      0     0.00                 -5.5                 8.25
fare_amount            float64      0     0.00               -490.0               6960.5
improvement_surcharge  float64      0     0.00                 -0.3                  0.3
mta_tax                float64      0     0.00                 -0.5                  0.5
passenger_count          int64  98352     7.18                  0.0                  8.0
p

In [7]:
import pandas as pd

url = ("https://github.com/DataTalksClub/nyc-tlc-data/releases/download/"
       "yellow/yellow_tripdata_2021-01.csv.gz")

counts = dict.fromkeys([
    "total_rows", "pickup_outside_jan_2021", "fare_amount < 0",
    "fare_amount == 0", "trip_distance < 0", "trip_distance == 0",
    "passenger_count == 0", "passenger_count is null", "duplicate_rows"], 0)
seen = set()

for chunk in pd.read_csv(url, chunksize=100_000):
    counts["total_rows"] += len(chunk)

    pu = pd.to_datetime(chunk["tpep_pickup_datetime"])
    counts["pickup_outside_jan_2021"] += int(((pu < "2021-01-01") | (pu >= "2021-02-01")).sum())

    counts["fare_amount < 0"] += int((chunk["fare_amount"] < 0).sum())
    counts["fare_amount == 0"] += int((chunk["fare_amount"] == 0).sum())
    counts["trip_distance < 0"] += int((chunk["trip_distance"] < 0).sum())
    counts["trip_distance == 0"] += int((chunk["trip_distance"] == 0).sum())
    counts["passenger_count == 0"] += int((chunk["passenger_count"] == 0).sum())
    counts["passenger_count is null"] += int(chunk["passenger_count"].isna().sum())

    hashes = pd.util.hash_pandas_object(chunk, index=False).values
    new = set(hashes) - seen
    counts["duplicate_rows"] += len(hashes) - len(new)
    seen |= new

problems = pd.DataFrame({"rows": counts})
problems["% of total"] = (problems["rows"] / counts["total_rows"] * 100).round(3)
print(problems)

/tmp/ipykernel_3141/281168446.py:12: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(url, chunksize=100_000):


                            rows  % of total
total_rows               1369765     100.000
pickup_outside_jan_2021       24       0.002
fare_amount < 0             6769       0.494
fare_amount == 0             642       0.047
trip_distance < 0              0       0.000
trip_distance == 0         19952       1.457
passenger_count == 0       26726       1.951
passenger_count is null    98352       7.180
duplicate_rows                 0       0.000


In [ ]:
def clean(df):
    df = df.copy()            
    removed = {}

    before = len(df)
    df = df[df["passenger_count"].notna()]
    removed["passenger_count is null"] = before - len(df)

    before = len(df)
    pu = pd.to_datetime(df["tpep_pickup_datetime"])
    df = df[(pu >= "2021-01-01") & (pu < "2021-02-01")]
    removed["pickup outside Jan 2021"] = before - len(df)

    before = len(df)
    df = df[df["fare_amount"] > 0]
    removed["fare_amount <= 0"] = before - len(df)

    return df, removed


total_before, total_after = 0, 0
total_removed = {}

for chunk in pd.read_csv(url, chunksize=100_000):
    cleaned, removed = clean(chunk)
    total_before += len(chunk)
    total_after += len(cleaned)
    for rule, n in removed.items():
        total_removed[rule] = total_removed.get(rule, 0) + n

print("rows before :", total_before)
print("rows after  :", total_after)
print("rows removed:", total_before - total_after)
print(pd.Series(total_removed, name="rows_removed"))

/tmp/ipykernel_3141/2843293231.py:24: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(url, chunksize=100_000):


rows before : 1369765
rows after  : 1264125
rows removed: 105640
passenger_count is null    98352
pickup outside Jan 2021       24
fare_amount <= 0            7264
Name: rows_removed, dtype: int64


In [9]:
decisions = pd.DataFrame([
    ["pickup outside Jan 2021", 24,     "DROP", "File is January only; dates like 2008 are wrong",      "Data engineering"],
    ["fare_amount <= 0",        7264,   "DROP", "A normal fare must be positive (cancel/refund/error)", "Business / finance"],
    ["passenger_count is null", 98352,  "DROP", "Row is incomplete (VendorID, payment_type also null)",  "Data source team"],
    ["passenger_count = 0",     26726,  "FLAG", "Driver may not have entered the count",                 "Business"],
    ["trip_distance = 0",       19952,  "FLAG", "Could be a cancelled trip or same pickup/dropoff",      "Business"],
    ["duplicate rows",          0,      "NONE", "No duplicates found",                                   "Data engineering"],
], columns=["problem", "rows", "decision", "why", "owner"])

print(decisions.to_string(index=False))

                problem  rows decision                                                  why              owner
pickup outside Jan 2021    24     DROP      File is January only; dates like 2008 are wrong   Data engineering
       fare_amount <= 0  7264     DROP A normal fare must be positive (cancel/refund/error) Business / finance
passenger_count is null 98352     DROP Row is incomplete (VendorID, payment_type also null)   Data source team
    passenger_count = 0 26726     FLAG                Driver may not have entered the count           Business
      trip_distance = 0 19952     FLAG     Could be a cancelled trip or same pickup/dropoff           Business
         duplicate rows     0     NONE                                  No duplicates found   Data engineering
